# B2.7 · Dynamic exploitation (DAST)

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.6 · Sandbox replication](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**.

| | |
|---|---|
| Tools used | OWASP ZAP, Nuclei, GLM-4.6, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Turn static findings into executable probes against the replica and separate confirmed from unconfirmed.

**Why a security engineer needs it.** A SAST finding is a hypothesis, and hypotheses get argued about instead of fixed. The control it builds is: stage 12: generate and run an actual exploit against the sandbox, so the finding is confirmed or dropped.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A finding becomes a fact the moment something other than a model says so. Driving the running application is how you get that second opinion — and the oracle you choose is what makes it worth having.

> **At CyberTravels.** A finding becomes a fact when something other than a model says so — here, a request to the replica's booking endpoint that returns another traveller's card details.

## 2 · The framework

```
   candidate finding
        |
        v
   payload --> running replica --> observed behaviour
                                        |
                             +----------v-----------+
                             |  ORACLE               |
                             |  did the state change?|
                             |  did the row appear?  |
                             +----------+-----------+
                                        v
                              confirmed | not confirmed

   the oracle is the whole value. "the model thinks so" is not one.
```

**Stage 12 — Dynamic exploitation.** The stage that converts an argument into a
fact.

Everything Phase 3 produced is a hypothesis: the code *looks* vulnerable and the
sink *appears* reachable. Hypotheses get argued about in triage meetings. An
executed exploit does not — either the probe achieved the effect or it did not.

Two things this stage produces that static analysis cannot:

- **Confirmation.** A finding that survives an exploit attempt is real,
  regardless of how the model felt about it.
- **Refutation.** A finding that fails is either not exploitable in this
  configuration or not real, and both are useful answers.

The discipline that makes it trustworthy is that the probe must assert a
**concrete effect** — rows returned that should not be, a file read outside the
root — not merely that the request did not error. "No exception" is the DAST
equivalent of a shape check, and B2.0 already established what those are worth.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Where it breaks — the probe that asserts nothing

The most common DAST bug: treating "the request succeeded" as confirmation. Both probes below hit the app and return 200-equivalents; only one of them proves anything.

## 4 · The stage, as a skill

A dynamic probe proves nothing unless its assertion can fail. The skill runs the probes against a live build with a control probe alongside, then re-runs them under a weak assertion so you can watch it flag the control too.

In [ ]:
# skills/appsec/dynamic-exploitation-probe/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: dynamic-exploitation-probe
description: >-
  Run probes against a live build to turn a static finding into a demonstrated
  one, with a control probe and an assertion strong enough to tell them apart.
  Use when confirming exploitability, or when a dynamic test reports everything
  as vulnerable.
allowed-tools: Read, Grep, Glob, Bash
---

# The assertion is the experiment

A dynamic probe proves nothing unless its assertion can fail. A weak assertion —
"the response contains rows", "the request did not error" — confirms the control
probe as readily as the attack, and a test that confirms everything has measured
nothing.

## When to use this

After static analysis has produced hypotheses, in a replica confirmed by an
isolation check, and before any finding is reported as exploitable.

## Procedure

**1 — State each hypothesis as a prediction.** Not "SQL injection in
`list_reports`" but "requesting one owner returns rows for more than one". A
hypothesis you cannot write as a prediction is not ready to test.

**2 — Include a control probe.** A well-formed, benign request whose result is
known. It is the thing that tells you your assertion discriminates.

**3 — Write the assertion against the prediction.** Count owners returned, not
whether rows came back. Read the file's contents, not whether a path resolved.
The difference between these is the entire value of the stage.

**4 — Run all probes, including the control, and record raw results.** Then
apply the assertion. Keep both: the raw result is what somebody re-reads when
they doubt the finding.

**5 — Demonstrate the weak assertion too.** Run the same probes under the loose
check and show it flagging the control. That comparison is what stops the next
person writing one.

**6 — Mark each hypothesis confirmed, refuted, or untested** — and never
"probably". A hypothesis whose probe could not run is untested, which is
different from refuted and much more common.

## Output contract

```json
{
  "hypotheses": [{"id": "str", "prediction": "str"}],
  "probes": [{"id": "str", "kind": "attack|control", "raw": "str"}],
  "assertions": [{"id": "str", "strong": true, "verdict": "confirmed|refuted|untested"}],
  "weak_assertion_run": {"flagged": ["str"], "includes_control": true}
}
```

## Failure modes

- **No control probe.** Nothing tells you the assertion discriminates.
- **Asserting on shape.** "Rows came back" is true for the legitimate query.
- **Recording untested as refuted.** They have different owners and different
  next steps.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/appsec/dynamic-exploitation-probe/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/appsec/dynamic-exploitation-probe/scripts/dynamic_exploitation_probe.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Run probes against a live build, and show what a weak assertion does to the control probe.

This is the executable half of the `dynamic-exploitation-probe` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import sqlite3
from dataclasses import dataclass, field

def build_app():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE reports(id INTEGER, owner TEXT, amount INT)")
    conn.executemany("INSERT INTO reports VALUES (?,?,?)",
                     [(1,"test-user-a",100),(2,"test-user-a",250),
                      (3,"test-user-b",900),(4,"test-user-c",50)])
    return conn

FILES = {"/srv/docs/readme.txt": "public readme",
         "/srv/secrets/token": "synthetic-not-real-9999"}

class App:
    """The replica under test. Two handlers, one of each bug class."""
    def __init__(self): self.conn = build_app()
    def list_reports(self, owner):
        q = "SELECT id, owner, amount FROM reports WHERE owner = '" + owner + "'"
        return self.conn.execute(q).fetchall()
    def fetch_doc(self, name):
        path = "/srv/docs/" + name
        parts = []
        for seg in path.split("/"):
            if seg in ("", "."): continue
            if seg == "..":
                if parts: parts.pop()
                continue
            parts.append(seg)
        real = "/" + "/".join(parts)
        return FILES.get(real, "not found")

app = App()
print("normal use:")
print("   list_reports('test-user-a') →", app.list_reports("test-user-a"))
print("   fetch_doc('readme.txt')     →", app.fetch_doc("readme.txt"))

@dataclass
class Probe:
    pid: str; cwe: str; target: str; payload: str
    effect: str                     # what must be TRUE for this to be exploitation

PROBES = [
 Probe("P-01","CWE-89","list_reports","x' OR '1'='1",
       "returns rows belonging to owners other than the one requested"),
 Probe("P-02","CWE-22","fetch_doc","../secrets/token",
       "returns file content from outside /srv/docs"),
 Probe("P-03","CWE-89","list_reports","test-user-a",
       "control probe — must NOT be flagged as exploitation"),
]

def run_probe(app, probe):
    if probe.target == "list_reports":
        rows = app.list_reports(probe.payload)
        owners = {r[1] for r in rows}
        exploited = len(owners) > 1
        return exploited, f"{len(rows)} rows, owners={sorted(owners)}"
    content = app.fetch_doc(probe.payload)
    exploited = content not in ("not found",) and "secrets" in probe.payload
    return exploited, f"returned {content!r}"

print(f"{'probe':7s}{'cwe':9s}{'exploited':11s}observation")
print("-" * 74)
results = {}
for p in PROBES:
    ok, obs = run_probe(app, p)
    results[p.pid] = ok
    print(f"{p.pid:7s}{p.cwe:9s}{str(ok):11s}{obs}")
assert results["P-01"] and results["P-02"] and not results["P-03"]

def weak_probe(app, probe):
    """Asserts only that nothing blew up."""
    try:
        if probe.target == "list_reports": app.list_reports(probe.payload)
        else: app.fetch_doc(probe.payload)
        return True, "request completed without error"
    except Exception as e:
        return False, f"raised {type(e).__name__}"

print(f"{'probe':7s}{'weak assertion':17s}{'effect assertion':18s}agreement")
print("-" * 66)
for p in PROBES:
    w, _ = weak_probe(app, p)
    s, _ = run_probe(app, p)
    print(f"{p.pid:7s}{str(w):17s}{str(s):18s}{'' if w == s else '← DISAGREE'}")
print("\nThe weak assertion confirms all three, including the control probe.")
print("A pipeline built on it reports 100% confirmation and means nothing by it.")

# Stage 12 output: findings promoted from hypothesis to confirmed, or dropped.
HYPOTHESES = [
 {"id":"F-01","cwe":"CWE-89","unit":"list_reports","status":"HYPOTHESIS","probe":"P-01"},
 {"id":"F-02","cwe":"CWE-22","unit":"fetch_doc",  "status":"HYPOTHESIS","probe":"P-02"},
 {"id":"F-03","cwe":"CWE-89","unit":"total",      "status":"HYPOTHESIS","probe":None},
]
def stage12(hypotheses, results):
    out = []
    for h in hypotheses:
        if h["probe"] is None:
            out.append({**h, "status": "UNVALIDATED",
                        "note": "no probe generated — cannot confirm or refute"})
        elif results.get(h["probe"]):
            out.append({**h, "status": "CONFIRMED",
                        "note": "exploit achieved the asserted effect on the replica"})
        else:
            out.append({**h, "status": "REFUTED",
                        "note": "probe ran; asserted effect did not occur"})
    return out

for f in stage12(HYPOTHESES, results):
    print(f"{f['id']}  {f['cwe']:9s}{f['status']:12s}{f['note']}")
confirmed = [f for f in stage12(HYPOTHESES, results) if f["status"] == "CONFIRMED"]
print(f"\n{len(confirmed)} confirmed by execution — these go to stages 13-15.")
print("UNVALIDATED is not a pass. It is a gap in probe generation, and it should")
print("appear in the report as one.")

## What you just proved

The SQL injection probe returns rows for three owners when one was requested, and the traversal probe returns the synthetic token from outside the document root; the control probe returns a single owner and is not flagged. The weak assertion confirms all three including the control. Stage 12 marks two findings CONFIRMED and one UNVALIDATED for having no probe.

## Your turn

Look at your DAST assertions. If any of them checks only for a non-error response, it is confirming findings it has not tested — and the control probe above is how you prove that in five minutes.

---

**Next → [B2.8 · Exploit chaining](https://spbreed.github.io/cyber-commons/lessons/B2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*